# 🚗 차량 손상 YOLO 학습 — Google Colab (GPU)

로컬 M1(MPS)보다 5~10배 빠른 무료 GPU(T4)로 학습합니다.

## 준비 (최초 1회)
1. 로컬에서 데이터셋 압축: `cd ~/Desktop/car-damage-analyzer/data && zip -r yolo_dataset.zip yolo_dataset`
2. `yolo_dataset.zip` 을 구글드라이브 `MyDrive/` 에 업로드
3. 상단 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
4. 이 노트북을 위에서부터 순서대로 실행

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. 라이브러리 설치 + 코드 클론

In [ ]:
!pip install ultralytics -q

import os

# 이미 클론된 경우 다시 클론하지 않음
if not os.path.exists('/content/CarCheck'):
    !git clone https://github.com/EunSeok-222/CarCheck.git

# 절대경로로 이동 (중복 실행해도 안전)
%cd /content/CarCheck

## 3. 구글드라이브 마운트 + 데이터셋 압축해제

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 드라이브의 zip → /content/CarCheck/data/ 로 압축해제 (절대경로)
import os
if not os.path.exists('/content/CarCheck/data/yolo_dataset'):
    !unzip -q /content/drive/MyDrive/yolo_dataset.zip -d /content/CarCheck/data/
    print('✅ 압축해제 완료')
else:
    print('✅ 이미 압축해제됨, 스킵')

!ls /content/CarCheck/data/yolo_dataset/images/train | wc -l
!ls /content/CarCheck/data/yolo_dataset/images/val | wc -l

## 4. Colab 경로용 yaml 생성

In [ ]:
yaml_text = '''path: /content/CarCheck/data/yolo_dataset
train: images/train
val:   images/val
nc: 4
names:
  0: Scratched
  1: Breakage
  2: Separated
  3: Crushed
'''
with open('data/car_damage_colab.yaml', 'w') as f:
    f.write(yaml_text)
print('✅ yaml 생성 완료')

## 5. 이어서 학습 (epoch 10 이후 +30 epoch)

epoch 10에서 mAP가 아직 상승 중이었으므로, **드라이브의 `best.pt`(epoch 10 가중치)에서 이어받아 30 epoch 더** 학습합니다.

- **처음 실행**: 드라이브 `best.pt`(epoch 10) 로드 → 추가 `EXTRA_EPOCHS` 학습 시작
- **재시작 시**: 이어서-학습 전용 체크포인트(`carcheck_ext_ckpt/last.pt`) 발견 시 중단 지점부터 자동 재개
- **매 epoch**: `last.pt` + `best.pt` + `args.yaml` → 드라이브 `carcheck_ext_ckpt/` 백업
- 더 돌리고 싶으면 아래 셀의 `EXTRA_EPOCHS` 값만 조정

In [ ]:
from ultralytics import YOLO
import shutil, os

# ── 설정 ──────────────────────────────────────────────
EXTRA_EPOCHS = 30          # epoch 10 이후 추가로 더 돌릴 epoch 수 (필요 시 조정)
RUN_NAME     = 'carcheck_ext'                              # 이어서-학습 전용 run
DRIVE_BEST   = '/content/drive/MyDrive/best.pt'            # 지금까지의 best (epoch 10)
DRIVE_CKPT   = '/content/drive/MyDrive/carcheck_ext_ckpt'  # 이어서-학습 전용 체크포인트
LOCAL_SAVE   = f'/content/CarCheck/runs/{RUN_NAME}'
# ──────────────────────────────────────────────────────

os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(f'{LOCAL_SAVE}/weights', exist_ok=True)

def save_checkpoint(trainer):
    """매 epoch마다 last.pt + args.yaml + best.pt → 드라이브 백업."""
    sd = str(trainer.save_dir)
    for fname, dst_name in [('weights/last.pt', 'last.pt'), ('args.yaml', 'args.yaml')]:
        src = f'{sd}/{fname}'
        if os.path.exists(src):
            shutil.copy(src, f'{DRIVE_CKPT}/{dst_name}')
    if os.path.exists(str(trainer.best)):
        shutil.copy(str(trainer.best), f'{DRIVE_CKPT}/best.pt')
    print(f'  💾 epoch {trainer.epoch + 1}: 체크포인트 저장')

drive_last = f'{DRIVE_CKPT}/last.pt'
drive_args = f'{DRIVE_CKPT}/args.yaml'

if os.path.exists(drive_last):
    # 이어서-학습(EXTRA_EPOCHS) 도중 Colab이 끊긴 경우 → 그 지점부터 재개
    print('✅ 이어서-학습 체크포인트 발견 → 중단 지점부터 재개')
    shutil.copy(drive_last, f'{LOCAL_SAVE}/weights/last.pt')
    if os.path.exists(drive_args):
        shutil.copy(drive_args, f'{LOCAL_SAVE}/args.yaml')
    model = YOLO(f'{LOCAL_SAVE}/weights/last.pt')
    model.add_callback('on_train_epoch_end', save_checkpoint)
    results = model.train(resume=True)
else:
    # epoch 10 best.pt 가중치에서 이어받아 추가 EXTRA_EPOCHS 학습 시작
    print(f'🔁 epoch 10 best.pt 에서 이어서 +{EXTRA_EPOCHS} epoch 추가 학습 시작')
    shutil.copy(DRIVE_BEST, '/content/CarCheck/continue_base.pt')
    model = YOLO('/content/CarCheck/continue_base.pt')   # ← epoch 10 가중치 로드
    model.add_callback('on_train_epoch_end', save_checkpoint)
    results = model.train(
        data='data/car_damage_colab.yaml',
        epochs=EXTRA_EPOCHS,
        imgsz=640,
        batch=16,
        device=0,
        patience=10,
        project='/content/CarCheck/runs',
        name=RUN_NAME,
        exist_ok=True,
    )

print('🎉 학습 완료')
print('Best mAP50(Box): ', results.results_dict.get('metrics/mAP50(B)', 0))
print('Best mAP50(Mask):', results.results_dict.get('metrics/mAP50(M)', 0))

best_pt = f'{str(results.save_dir)}/weights/best.pt'
if os.path.exists(best_pt):
    shutil.copy(best_pt, DRIVE_BEST)
    print('✅ best.pt → 드라이브 최종 저장 완료')

In [ ]:
# ── 성능 향상 그래프 (이어서-학습 mAP 곡선) ──────────────
# 한글 폰트 설치 (Colab 기본엔 한글 폰트가 없어 □□□ 깨짐 방지)
!apt-get install -y fonts-nanum -q > /dev/null 2>&1
import matplotlib.font_manager as fm
_font = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(_font):
    fm.fontManager.addfont(_font)

import pandas as pd
import matplotlib.pyplot as plt
import glob, os

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 이어서-학습 run 의 results.csv 자동 탐색
csv_path = f'{str(results.save_dir)}/results.csv'
if not os.path.exists(csv_path):
    cands = glob.glob('/content/CarCheck/runs/carcheck_ext*/results.csv')
    csv_path = cands[0] if cands else None

if csv_path and os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()          # 컬럼명 공백 제거

    epochs   = df['epoch']
    box_map  = df['metrics/mAP50(B)']
    mask_map = df['metrics/mAP50(M)']

    # 이어서-학습 시작점(= 직전 10-epoch 최종 성능)과 발표 목표선
    START_BOX,  START_MASK  = 0.177, 0.120        # +30 학습 직전(epoch 10) 성능
    TARGET_BOX, TARGET_MASK = 0.239, 0.212        # 기존 발표 best (넘어야 할 목표)

    plt.style.use('dark_background')
    plt.rcParams['font.family'] = 'NanumGothic'   # style.use 뒤 재설정
    fig, ax = plt.subplots(figsize=(9, 5.2), dpi=130)
    fig.patch.set_facecolor('#0d1b2a')
    ax.set_facecolor('#0d1b2a')

    ax.plot(epochs, box_map,  color='#38bdf8', lw=2.4, marker='o', ms=4, label='Box mAP50')
    ax.plot(epochs, mask_map, color='#4ade80', lw=2.4, marker='o', ms=4, label='Mask mAP50')

    # 시작점(이어서 학습 직전) / 목표선
    ax.axhline(START_BOX,   color='#38bdf8', ls=':', lw=1.2, alpha=0.6)
    ax.axhline(TARGET_BOX,  color='#f59e0b', ls='--', lw=1.5, alpha=0.9)
    ax.text(epochs.iloc[-1], TARGET_BOX + 0.004, f'기존 best {TARGET_BOX}',
            color='#f59e0b', fontsize=9, ha='right')

    # 최종 성능 주석
    fb, fm_ = box_map.iloc[-1], mask_map.iloc[-1]
    ax.annotate(f'{fb:.3f}', (epochs.iloc[-1], fb), color='#38bdf8',
                fontsize=10, fontweight='bold', ha='left', va='bottom')
    ax.annotate(f'{fm_:.3f}', (epochs.iloc[-1], fm_), color='#4ade80',
                fontsize=10, fontweight='bold', ha='left', va='top')

    ax.set_title('이어서 학습(+30 epoch) 성능 향상 곡선', color='white', fontsize=13, pad=12)
    ax.set_xlabel('Epoch (이어서-학습 구간)', color='#cbd5e1')
    ax.set_ylabel('mAP50', color='#cbd5e1')
    ax.tick_params(colors='#94a3b8')
    ax.grid(True, alpha=0.15)
    ax.legend(facecolor='#1e293b', edgecolor='none', labelcolor='white', loc='lower right')
    for s in ax.spines.values():
        s.set_color('#334155')

    out_png = '/content/CarCheck/runs/carcheck_ext_improvement.png'
    plt.tight_layout()
    plt.savefig(out_png, facecolor=fig.get_facecolor(), bbox_inches='tight')
    plt.show()

    # 향상폭 요약 출력
    print('\n── 성능 향상 요약 ──')
    print(f'Box  mAP50: {START_BOX:.3f} → {fb:.3f}  ({(fb-START_BOX):+.3f})  | 기존 best {TARGET_BOX} {"✅ 돌파" if fb>=TARGET_BOX else "❌ 미달"}')
    print(f'Mask mAP50: {START_MASK:.3f} → {fm_:.3f}  ({(fm_-START_MASK):+.3f})  | 기존 best {TARGET_MASK} {"✅ 돌파" if fm_>=TARGET_MASK else "❌ 미달"}')
    print(f'📁 그래프 저장: {out_png}')

    # 드라이브에도 백업
    try:
        import shutil
        shutil.copy(out_png, '/content/drive/MyDrive/carcheck_ext_improvement.png')
        print('✅ 그래프 → 드라이브 백업 완료')
    except Exception as e:
        print('그래프 드라이브 백업 스킵:', e)
else:
    print('❌ results.csv 를 찾지 못했습니다. 학습 셀(위)이 정상 완료됐는지 확인하세요.')

## 6. 결과 그래프 + best.pt 드라이브에 저장

In [ ]:
from IPython.display import Image as IPImage, display
import glob

# 실제 저장된 results.png 자동 탐색
result_pngs = glob.glob('/content/CarCheck/runs/segment/**/results.png', recursive=True)
if result_pngs:
    display(IPImage(result_pngs[0]))
else:
    print('results.png 없음')

In [ ]:
from google.colab import files
import glob

# best.pt 자동 탐색 후 다운로드
best_pts = glob.glob('/content/CarCheck/runs/segment/**/best.pt', recursive=True)
if best_pts:
    print(f'📁 best.pt 경로: {best_pts[0]}')
    files.download(best_pts[0])
else:
    print('❌ best.pt 없음 — 드라이브에서 확인하세요: MyDrive/best.pt')

## 학습 후 로컬 적용

드라이브/다운로드받은 `best.pt` 를 로컬 `car-damage-analyzer/models/best.pt` 에 덮어쓰면
Streamlit 앱이 자동으로 새 모델을 사용합니다.